In [ ]:
'''
김싸피가 속한 마케팅 부서는 고객의 누적 가치를 기반으로
특정 기준 이상을 만족하는 VIP 고객에게 특별 쿠폰을 발송하려 합니다.
고객 구매 내역에서 Customer Lifetime Value를 합산하고,
기준 금액 이상인 경우 "alert": "yes"로 표시해 CSV로 저장하세요.
사용 데이터셋 : Global Customer Loyalty Program
https://www.kaggle.com/datasets/imadali595/global-customer-loyalty-program?utm_source=chatgpt.com
'''

In [ ]:
# 요구 사항
'''
기준 금액은 50,000으로 고정
groupby → sum → reset_index → merge 순서로 구성
alert 컬럼은 "yes" 또는 "no"로 구분
'''

In [1]:
import os
import pandas as pd

DATA_PATH = "../data/CustomerLoyaltyProgram.csv"
OUTPUT_DIR = "../output/curated"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "alert_lifetime_summary.csv")
THRESHOLD = 50000

In [2]:
def extract_customers(path: str) -> pd.DataFrame:
    """
    Extract 단계: 원천 CSV 로드
    pandas에서 CSV를 읽는 함수 사용
    """
    df = pd.read_csv(DATA_PATH)
    return df

In [3]:
def transform_alert_summary(df: pd.DataFrame, threshold: int) -> pd.DataFrame:
    """
    Transform 단계:
    - groupby -> sum -> reset_index -> merge 순서 사용
    - 고객별 CLV 합계 계산
    - alert 컬럼("yes"/"no") 생성
    """

    # (선택) CLV 결측치 처리: NaN을 0으로 치환
    df["Customer Lifetime Value"] = df["Customer Lifetime Value"].fillna(0)

    # 1) 고객별 CLV 합계
    summary = (
        df.groupby("Loyalty#")["Customer Lifetime Value"]
          .sum()
          .reset_index()
    )

    # 2) 컬럼명 변경
    summary = summary.rename(columns={"Customer Lifetime Value": "TotalLifetimeValue"})

    # 3) 고객명 매핑 테이블 생성 (Loyalty# 기준 중복 제거)
    name_map = df.drop_duplicates("Loyalty#")[["Loyalty#", "Customer Name"]]

    # 4) merge
    summary = summary.merge(name_map, on="Loyalty#", how="left")

    # 5) alert 컬럼 생성 (threshold 이상이면 "yes", 아니면 "no")
    summary["alert"] = summary["TotalLifetimeValue"].ge(threshold).map({True: "yes", False: "no"})

    # 6) 컬럼 순서 정리
    summary = summary[["Loyalty#", "Customer Name", "TotalLifetimeValue", "alert"]]

    return summary

In [4]:
def load_csv(df: pd.DataFrame, out_path: str):
    """
    Load 단계: 디렉토리 생성 후 CSV 저장
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)

In [5]:
df = extract_customers(DATA_PATH)
summary_df = transform_alert_summary(df, THRESHOLD)
load_csv(summary_df, OUTPUT_PATH)

print("Job1 완료: 요약 테이블 생성")
print(summary_df.sort_values("TotalLifetimeValue", ascending=False).head())

Job1 완료: 요약 테이블 생성
       Loyalty#    Customer Name  TotalLifetimeValue alert
51619  834417.0   Rocio Millisor           249976.14   yes
29265  514396.0    Teodoro Fiume           193856.28   yes
57091  912633.0  Aleisha Andrino           189820.81   yes
47895  781219.0  Elenore Restifo           188964.84   yes
61334  972502.0  Johnette Zutell           177232.92   yes
